# Lab 9 — Transformer NMT with Pretrained Embeddings**Task.** Same English -> Hindi data as Lab 8 (ILCI subset of BPCC, same four length bins),but the model is now a **Transformer**, and it is trained twice with two differentinitialisations of the embedding table:| system | tokenisation | embeddings ||---|---|---|| **A** | word-level | FastText pretrained word vectors (300d) || **B** | BPEmb subwords | BPEmb pretrained subword vectors (300d) |Both are evaluated with sacreBLEU, reported per bin, and compared against each other.Because the pretrained vectors are 300-dimensional, the Transformer is built with`hidden_size = word_vec_size = 300` and 6 attention heads (300 / 6 = 50 per head).

## 1. Environment

In [ ]:
import os, sys, re, json, gzip, io, random, shutil, platform, subprocessfrom collections import defaultdictimport numpy as npimport requestsimport torchimport sacrebleurandom.seed(42)np.random.seed(42)torch.manual_seed(42)print("python  :", sys.version.split()[0], "|", platform.system(), platform.machine())print("torch   :", torch.__version__)print("cuda    :", torch.cuda.is_available(),      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no NVIDIA GPU")USE_GPU = torch.cuda.is_available()IS_WIN  = platform.system() == "Windows"ROOT = "nmt9"for d in ["data", "emb", "models", "preds"]:    os.makedirs(f"{ROOT}/{d}", exist_ok=True)

In [ ]:
def onmt_cmd(name):    """Resolve onmt_build_vocab / onmt_train / onmt_translate on any platform."""    exe = shutil.which(f"onmt_{name}")    return [exe] if exe else [sys.executable, "-m", f"onmt.bin.{name}"]def run(cmd):    """Run a subprocess and stream its output into the notebook as it goes."""    print("$", " ".join(str(c) for c in cmd), flush=True)    p = subprocess.Popen([str(c) for c in cmd], stdout=subprocess.PIPE,                         stderr=subprocess.STDOUT, text=True, bufsize=1,                         encoding="utf-8", errors="replace")    for line in p.stdout:        print(line, end="")    p.wait()    if p.returncode != 0:        raise RuntimeError(f"command exited with code {p.returncode}")import onmtprint("OpenNMT-py:", onmt.__version__)

In [ ]:
# ----- Run size -------------------------------------------------------------# This notebook trains TWO Transformers, so the CPU budget is half of what a single# model would get. Raise these if you are running on CUDA and want stronger numbers.if USE_GPU:    TRAIN_PER_BIN, TRAIN_STEPS, LAYERS = 15000, 6000, 4else:    TRAIN_PER_BIN, TRAIN_STEPS, LAYERS = 2500, 1200, 2VALID_PER_BIN = 500TEST_PER_BIN  = 500VALID_STEPS   = max(TRAIN_STEPS // 6, 200)EMB_DIM       = 300        # fixed by both pretrained embedding setsHEADS         = 6          # 300 / 6 = 50 dims per headBPEMB_VS      = 10000      # BPEmb vocabulary size per languageFT_MAX_LINES  = 400_000    # how deep to read into the frequency-sorted FastText fileprint(f"device {'cuda' if USE_GPU else 'cpu'} | {TRAIN_PER_BIN} sent/bin | "      f"{TRAIN_STEPS} steps x 2 systems | {LAYERS}-layer Transformer")

## 2. Data — same corpus and the same four bins as Lab 8

In [ ]:
from datasets import get_dataset_config_names, load_datasetALL_CONFIGS = get_dataset_config_names("ai4bharat/BPCC")for c in [c for c in ALL_CONFIGS if "ilci" in c.lower() or "human" in c.lower()]:    print("  ", c)

In [ ]:
CONFIG   = "bpcc-human"      # <-- match the ILCI config printed aboveSRC_COL  = "src"TGT_COL  = "tgt"LANG_COL = "tgt_lang"TGT_LANG = "hin_Deva"raw = load_dataset("ai4bharat/BPCC", CONFIG)split_name = list(raw.keys())[0]ds = raw[split_name]print("columns:", ds.column_names)if LANG_COL and LANG_COL in ds.column_names:    ds = ds.filter(lambda r: r[LANG_COL] == TGT_LANG)pairs = [(e.strip(), h.strip())         for e, h in zip(ds[SRC_COL], ds[TGT_COL])         if e and h and e.strip() and h.strip()]print(f"{len(pairs):,} pairs")

In [ ]:
def length_bin(sentence):    # Bins are defined on the English source length in whitespace tokens.    # bin2 is closed at 20 and bin4 opens at 30 so that no length is left unassigned.    n = len(sentence.split())    if n <= 10:  return 1    if n <= 20:  return 2    if n <= 29:  return 3    return 4by_bin = defaultdict(list)for e, h in pairs:    by_bin[length_bin(e)].append((e, h))train, valid, test = [], [], defaultdict(list)for b in (1, 2, 3, 4):    bucket = by_bin[b][:]    random.shuffle(bucket)    n_test  = min(TEST_PER_BIN,  len(bucket) // 10)    n_valid = min(VALID_PER_BIN, len(bucket) // 10)    n_train = min(TRAIN_PER_BIN, len(bucket) - n_test - n_valid)    test[b] = bucket[:n_test]    valid  += bucket[n_test:n_test + n_valid]    train  += bucket[n_test + n_valid:n_test + n_valid + n_train]random.shuffle(train)random.shuffle(valid)print(f"train {len(train):,} | valid {len(valid):,}")for b in (1, 2, 3, 4):    print(f"  bin{b}: corpus {len(by_bin[b]):>7,}   test {len(test[b])}")

## 3. Two tokenisations**System A** needs surface words, because FastText vectors are keyed by words.**System B** needs BPEmb subwords, because that is the unit BPEmb provides vectors for.

In [ ]:
# ---- A: word-level -------------------------------------------------------TOKEN_RE = re.compile(r"\w+|[^\w\s]", re.UNICODE)def word_tok(text):    return TOKEN_RE.findall(text)def word_detok(tokens):    s = " ".join(tokens)    s = re.sub(r"\s+([।,.!?;:'\")\]])", r"\1", s)   # no space before closing punctuation    s = re.sub(r"([(\[\"])\s+", r"\1", s)            # no space after opening punctuation    return s.strip()print(word_tok("The weather changed, very quickly!"))print(word_detok(word_tok("पहाड़ों में मौसम बहुत तेजी से बदल गया।")))

In [ ]:
# ---- B: BPEmb subwords ---------------------------------------------------# First call downloads the model and vectors into ~/.cache/bpemb (a few hundred MB).from bpemb import BPEmbbpemb_en = BPEmb(lang="en", vs=BPEMB_VS, dim=EMB_DIM)bpemb_hi = BPEmb(lang="hi", vs=BPEMB_VS, dim=EMB_DIM)def bpe_detok(tokens):    # BPEmb marks word starts with the sentencepiece meta-symbol U+2581.    return "".join(tokens).replace("\u2581", " ").strip()demo = bpemb_en.encode("The weather changed very quickly.")print(demo)print(bpe_detok(demo))print(bpemb_hi.encode("पहाड़ों में मौसम बहुत तेजी से बदल गया।"))

In [ ]:
def write_tokenised(pairs_list, prefix, en_fn, hi_fn):    with open(f"{prefix}.en", "w", encoding="utf-8", newline="\n") as fe, \         open(f"{prefix}.hi", "w", encoding="utf-8", newline="\n") as fh:        for e, h in pairs_list:            fe.write(" ".join(en_fn(e)) + "\n")            fh.write(" ".join(hi_fn(h)) + "\n")# System A fileswrite_tokenised(train, f"{ROOT}/data/trainA", word_tok, word_tok)write_tokenised(valid, f"{ROOT}/data/validA", word_tok, word_tok)# System B fileswrite_tokenised(train, f"{ROOT}/data/trainB", bpemb_en.encode, bpemb_hi.encode)write_tokenised(valid, f"{ROOT}/data/validB", bpemb_en.encode, bpemb_hi.encode)# Test sources, one file per bin per systemfor b in (1, 2, 3, 4):    with open(f"{ROOT}/data/testA.bin{b}.en", "w", encoding="utf-8", newline="\n") as fa, \         open(f"{ROOT}/data/testB.bin{b}.en", "w", encoding="utf-8", newline="\n") as fb:        for e, _ in test[b]:            fa.write(" ".join(word_tok(e)) + "\n")            fb.write(" ".join(bpemb_en.encode(e)) + "\n")print(f"{len(os.listdir(f'{ROOT}/data'))} files written to {ROOT}/data")

## 4. VocabulariesBoth systems need their vocabulary built before the embedding tables can be aligned to it,so the vocab step runs first and training follows afterwards.

In [ ]:
GPU_BLOCK = "world_size: 1\ngpu_ranks: [0]\n" if USE_GPU else ""WORKERS   = "num_workers: 2\n" if (USE_GPU and not IS_WIN) else "num_workers: 0\n"def write_config(tag, extra=""):    """Emit an OpenNMT-py YAML config for one system."""    cfg = f"""save_data: {ROOT}/data/vocab{tag}src_vocab: {ROOT}/data/vocab{tag}.srctgt_vocab: {ROOT}/data/vocab{tag}.tgtsrc_vocab_size: 32000tgt_vocab_size: 32000overwrite: Truedata:    corpus_1:        path_src: {ROOT}/data/train{tag}.en        path_tgt: {ROOT}/data/train{tag}.hi    valid:        path_src: {ROOT}/data/valid{tag}.en        path_tgt: {ROOT}/data/valid{tag}.hisave_model: {ROOT}/models/tf{tag}save_checkpoint_steps: {max(TRAIN_STEPS // 3, 400)}keep_checkpoint: 3train_steps: {TRAIN_STEPS}valid_steps: {VALID_STEPS}report_every: 100{GPU_BLOCK}{WORKERS}# --- Transformer, sized to the 300-d pretrained vectors (300 / {HEADS} heads = {EMB_DIM // HEADS}) ---encoder_type: transformerdecoder_type: transformerlayers: {LAYERS}heads: {HEADS}hidden_size: {EMB_DIM}word_vec_size: {EMB_DIM}transformer_ff: 1200position_encoding: truedropout: [0.1]attention_dropout: [0.1]# --- optimisation: Adam with the noam inverse-sqrt schedule ---optim: adamadam_beta2: 0.998decay_method: noamwarmup_steps: {min(1000, TRAIN_STEPS // 4)}learning_rate: 2max_grad_norm: 0batch_size: {4096 if USE_GPU else 1024}batch_type: tokensaccum_count: [{4 if USE_GPU else 1}]label_smoothing: 0.1param_init: 0param_init_glorot: truenormalization: tokens{extra}"""    path = f"{ROOT}/tf{tag}.yaml"    with open(path, "w", encoding="utf-8") as f:        f.write(cfg)    return path# Vocab-only pass for both systems (no embeddings referenced yet).for tag in ("A", "B"):    p = write_config(tag)    run(onmt_cmd("build_vocab") + ["-config", p, "-n_sample", "-1"])    print("built vocab for system", tag, "\n")

In [ ]:
# OpenNMT-py prepends four special tokens to the vocabulary file, in this order.# The pretrained embedding matrix must use the same row order, or every lookup is wrong.SPECIALS = ["<unk>", "<blank>", "<s>", "</s>"]def read_vocab(path):    toks = []    with open(path, encoding="utf-8") as f:        for line in f:            parts = line.rstrip("\n").split("\t")            if parts[0]:                toks.append(parts[0])    return SPECIALS + [t for t in toks if t not in SPECIALS]vocabA_src = read_vocab(f"{ROOT}/data/vocabA.src")vocabA_tgt = read_vocab(f"{ROOT}/data/vocabA.tgt")vocabB_src = read_vocab(f"{ROOT}/data/vocabB.src")vocabB_tgt = read_vocab(f"{ROOT}/data/vocabB.tgt")for n, v in [("A.src", vocabA_src), ("A.tgt", vocabA_tgt),             ("B.src", vocabB_src), ("B.tgt", vocabB_tgt)]:    print(f"{n}: {len(v):>6,} types   first 6: {v[:6]}")

## 5. System A embeddings — FastTextThe full `cc.*.300.vec.gz` files are several GB. They are sorted by frequency, so the fileis streamed over HTTP and parsed line by line, keeping only vectors for words that areactually in our vocabulary and stopping at `FT_MAX_LINES`. Nothing is written to disk.If your connection drops mid-stream, re-run the cell — it starts over cleanly.

In [ ]:
FASTTEXT_URLS = {    "en": "https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.en.300.vec.gz",    "hi": "https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.hi.300.vec.gz",}def stream_fasttext(url, wanted, max_lines=FT_MAX_LINES, dim=EMB_DIM):    """Pull vectors for `wanted` out of a remote .vec.gz without downloading it whole."""    wanted = set(wanted)    found = {}    with requests.get(url, stream=True, timeout=180) as r:        r.raise_for_status()        r.raw.decode_content = False          # the payload is gzip, not a transfer encoding        gz = gzip.GzipFile(fileobj=r.raw)        reader = io.TextIOWrapper(gz, encoding="utf-8", errors="ignore")        reader.readline()                     # header: "<n_words> <dim>"        for i, line in enumerate(reader):            if i >= max_lines or len(found) == len(wanted):                break            sp = line.rstrip().split(" ")            if sp[0] in wanted and len(sp) == dim + 1:                found[sp[0]] = np.asarray(sp[1:], dtype=np.float32)    return foundft_en = stream_fasttext(FASTTEXT_URLS["en"], vocabA_src)print(f"English: {len(ft_en):,} / {len(vocabA_src):,} vocabulary types covered")ft_hi = stream_fasttext(FASTTEXT_URLS["hi"], vocabA_tgt)print(f"Hindi  : {len(ft_hi):,} / {len(vocabA_tgt):,} vocabulary types covered")

In [ ]:
def build_emb_matrix(vocab, lookup, dim=EMB_DIM, lowercase_backoff=False):    """Row i of the result is the vector for vocab[i]; misses get a small random vector."""    mat = np.random.normal(0, 0.1, (len(vocab), dim)).astype(np.float32)    mat[1] = 0.0                                   # <blank> (padding) stays at zero    hits = 0    for i, tok in enumerate(vocab):        vec = lookup.get(tok)        if vec is None and lowercase_backoff:            vec = lookup.get(tok.lower())        if vec is not None:            mat[i] = vec            hits += 1    return torch.from_numpy(mat), hitsembA_src, hs = build_emb_matrix(vocabA_src, ft_en, lowercase_backoff=True)embA_tgt, ht = build_emb_matrix(vocabA_tgt, ft_hi)torch.save(embA_src, f"{ROOT}/emb/ftA.enc.pt")torch.save(embA_tgt, f"{ROOT}/emb/ftA.dec.pt")print(f"encoder matrix {tuple(embA_src.shape)} — {hs/len(vocabA_src):.1%} initialised from FastText")print(f"decoder matrix {tuple(embA_tgt.shape)} — {ht/len(vocabA_tgt):.1%} initialised from FastText")

## 6. System B embeddings — BPEmbBPEmb ships its vectors alongside the segmentation model, so there is no download stephere and coverage should be near-total: every token in the vocabulary came out of the sameBPEmb segmenter that produced the vectors.

In [ ]:
def bpemb_lookup(bpemb_model):    kv = bpemb_model.emb    return {w: kv[w] for w in kv.index_to_key}embB_src, hs = build_emb_matrix(vocabB_src, bpemb_lookup(bpemb_en))embB_tgt, ht = build_emb_matrix(vocabB_tgt, bpemb_lookup(bpemb_hi))torch.save(embB_src, f"{ROOT}/emb/bpeB.enc.pt")torch.save(embB_tgt, f"{ROOT}/emb/bpeB.dec.pt")print(f"encoder matrix {tuple(embB_src.shape)} — {hs/len(vocabB_src):.1%} initialised from BPEmb")print(f"decoder matrix {tuple(embB_tgt.shape)} — {ht/len(vocabB_tgt):.1%} initialised from BPEmb")

## 7. Training both systemsThe embedding paths are added to each config and the vocab step is re-run (it is cheap andkeeps the config file and the vocab in sync), then each Transformer trains in turn.

In [ ]:
extraA = f"""pre_word_vecs_enc: {ROOT}/emb/ftA.enc.ptpre_word_vecs_dec: {ROOT}/emb/ftA.dec.ptfreeze_word_vecs_enc: falsefreeze_word_vecs_dec: false"""extraB = f"""pre_word_vecs_enc: {ROOT}/emb/bpeB.enc.ptpre_word_vecs_dec: {ROOT}/emb/bpeB.dec.ptfreeze_word_vecs_enc: falsefreeze_word_vecs_dec: false"""CFG_A = write_config("A", extra=extraA)CFG_B = write_config("B", extra=extraB)print(open(CFG_A, encoding="utf-8").read())

In [ ]:
# ---- System A: Transformer + FastText ----run(onmt_cmd("build_vocab") + ["-config", CFG_A, "-n_sample", "-1"])run(onmt_cmd("train") + ["-config", CFG_A])

In [ ]:
# ---- System B: Transformer + BPEmb ----run(onmt_cmd("build_vocab") + ["-config", CFG_B, "-n_sample", "-1"])run(onmt_cmd("train") + ["-config", CFG_B])

In [ ]:
def latest_ckpt(tag):    fs = [f for f in os.listdir(f"{ROOT}/models") if f.startswith(f"tf{tag}_step_")]    fs.sort(key=lambda f: int(f.split("_step_")[1].split(".pt")[0]))    return f"{ROOT}/models/{fs[-1]}"MODEL_A, MODEL_B = latest_ckpt("A"), latest_ckpt("B")print("A:", MODEL_A)print("B:", MODEL_B)

## 8. Decoding and bin-wise BLEU

In [ ]:
gpu_flag = ["-gpu", "0"] if USE_GPU else []for tag, model in (("A", MODEL_A), ("B", MODEL_B)):    for b in (1, 2, 3, 4):        run(onmt_cmd("translate") + [            "-model", model,            "-src", f"{ROOT}/data/test{tag}.bin{b}.en",            "-output", f"{ROOT}/preds/pred{tag}.bin{b}.hi",            "-beam_size", "5",            "-batch_size", "16",        ] + gpu_flag)    print(f"--- decoded system {tag} ---\n")

In [ ]:
def load_hyps(tag, b, detok):    with open(f"{ROOT}/preds/pred{tag}.bin{b}.hi", encoding="utf-8") as f:        return [detok(line.strip().split()) for line in f]hyps = {"A": {}, "B": {}}for b in (1, 2, 3, 4):    hyps["A"][b] = load_hyps("A", b, word_detok)    hyps["B"][b] = load_hyps("B", b, bpe_detok)print("bin2, first two sentences\n")for i in range(2):    print(" SRC:", test[2][i][0])    print(" REF:", test[2][i][1])    print(" A  :", hyps["A"][2][i])    print(" B  :", hyps["B"][2][i], "\n")

In [ ]:
scores = {"A": [], "B": []}overall = {}for tag in ("A", "B"):    allh, allr = [], []    for b in (1, 2, 3, 4):        refs = [h for _, h in test[b]]        hyp  = hyps[tag][b]        bleu = sacrebleu.corpus_bleu(hyp, [refs])        scores[tag].append({            "bin": b,            "n": len(refs),            "avg_src_len": round(sum(len(e.split()) for e, _ in test[b]) / len(test[b]), 1),            "BLEU": round(bleu.score, 2),            "p1": round(bleu.precisions[0], 1),            "p4": round(bleu.precisions[3], 1),            "len_ratio": round(bleu.sys_len / bleu.ref_len, 3),        })        allh += hyp        allr += refs    overall[tag] = sacrebleu.corpus_bleu(allh, [allr]).scorehdr = f"{'bin':<6}{'n':>5}{'avg len':>9}{'A BLEU':>9}{'B BLEU':>9}{'B - A':>8}{'A p4':>7}{'B p4':>7}"print(hdr)print("-" * len(hdr))for i, b in enumerate((1, 2, 3, 4)):    a, bb = scores["A"][i], scores["B"][i]    print(f"bin{b:<3}{a['n']:>5}{a['avg_src_len']:>9}{a['BLEU']:>9}{bb['BLEU']:>9}"          f"{bb['BLEU'] - a['BLEU']:>8.2f}{a['p4']:>7}{bb['p4']:>7}")print("-" * len(hdr))print(f"{'ALL':<6}{'':>5}{'':>9}{overall['A']:>9.2f}{overall['B']:>9.2f}"      f"{overall['B'] - overall['A']:>8.2f}")with open(f"{ROOT}/results_lab9.json", "w", encoding="utf-8") as f:    json.dump({"systems": scores, "overall": {k: round(v, 2) for k, v in overall.items()}},              f, indent=2)

## 9. Bin-wise analysisThe cell below turns the table above into statements. Run it and use the output as thebasis for your written analysis.

In [ ]:
A = {r["bin"]: r for r in scores["A"]}B = {r["bin"]: r for r in scores["B"]}print("Observations\n" + "=" * 70)print(f"1. System A (FastText, word-level): bin1 {A[1]['BLEU']}, bin2 {A[2]['BLEU']}, "      f"bin3 {A[3]['BLEU']}, bin4 {A[4]['BLEU']} | overall {overall['A']:.2f}")print(f"   System B (BPEmb, subword)      : bin1 {B[1]['BLEU']}, bin2 {B[2]['BLEU']}, "      f"bin3 {B[3]['BLEU']}, bin4 {B[4]['BLEU']} | overall {overall['B']:.2f}")better = "B" if overall["B"] > overall["A"] else "A"print(f"\n2. Overall, system {better} wins by {abs(overall['B'] - overall['A']):.2f} BLEU.")gaps = {b: B[b]["BLEU"] - A[b]["BLEU"] for b in (1, 2, 3, 4)}widest = max(gaps, key=lambda b: abs(gaps[b]))print(f"3. The gap between the two systems is widest in bin{widest} "      f"({gaps[widest]:+.2f} BLEU) and narrowest in "      f"bin{min(gaps, key=lambda b: abs(gaps[b]))}.")for tag, S in (("A", A), ("B", B)):    drop = S[1]["BLEU"] - S[4]["BLEU"]    print(f"4{tag}. System {tag} loses {drop:.2f} BLEU going from bin1 to bin4; "          f"4-gram precision {S[1]['p4']} -> {S[4]['p4']}, "          f"length ratio {S[1]['len_ratio']} -> {S[4]['len_ratio']}.")print(f"\n5. Embedding coverage: the FastText matrices were initialised for a fraction of "      f"the word vocabulary (printed in section 5), while BPEmb covered nearly all of its "      f"subword vocabulary (section 6). Quote both figures in the write-up — they are the "      f"main reason the two systems differ.")

### Points to address in the write-upCheck each of these against your own numbers above rather than assuming the direction.**FastText vs BPEmb**- *Coverage.* FastText is keyed by whole words, so every morphological variant of a Hindi  word is a separate entry and anything outside the frequency window you streamed falls  back to a random vector. BPEmb's vocabulary is closed by construction — the segmenter and  the vectors come from the same model — so there is effectively no OOV. Compare the two  coverage percentages printed in sections 5 and 6.- *Vocabulary size and data efficiency.* The word-level system carries a much larger  embedding table trained on the same number of sentences, so each row gets fewer updates.  The subword table is small and every row is seen often.- *Hindi morphology.* Case marking and compounding generate many surface forms from one  stem. Subwords share parameters across those forms; whole-word vectors do not.- *What FastText still brings.* Its vectors are trained on far more text than BPEmb's, so  for frequent words the word-level system starts from a better-informed point — which is  why its advantage, if any, tends to show up in the short bins where common vocabulary  dominates.**Length effects, versus Lab 8**- The Transformer's self-attention reaches any source position in one step, so the long-bin  degradation should be gentler here than with the Lab 8 BiLSTM. Compare bin4 across the  two labs directly — `nmt/results_lab8.json` and `nmt9/results_lab9.json` hold both tables.- Residual BLEU loss in bin3 and bin4 is mostly long-distance reordering (English SVO to  Hindi SOV across multiple clauses) and the n-gram brittleness of BLEU on long outputs,  not an inability to see the source.- Watch the length ratio column in bin4. A ratio well under 1 means truncation; well over 1  usually means the decoder fell into repetition.